In [1]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Dense, GlobalAveragePooling1D
import numpy as np

print("TensorFlow Version:", tf.__version__)

# =====================================================================
# 1. DATASET SIMULASI RELEVAN (DOMAIN: DATA ANALYST)
# =====================================================================
# Dataset diperbanyak secara seimbang agar model bisa belajar pola dengan akurat
jawaban_train = [
    # KELAS 0: WEAK (Klaim kosong, tanpa tools, tanpa metode STAR)
    "saya hanya bisa membuat dashboard sederhana saja di proyek kemarin.",
    "pengalaman saya adalah menganalisis data penjualan toko secara manual.",
    "saya cuma melihat grafik data yang ada di excel saja sih kak.",
    "saya tahu apa itu query SQL tapi belum pernah mempraktikkannya.",

    # KELAS 1: AVERAGE (Menyebutkan tools dan konteks, tapi struktur STAR menggantung)
    "saya membangun dashboard penjualan menggunakan tableau untuk visualisasi data kuliah.",
    "saya melakukan proses data cleaning pada dataset kotor menggunakan library pandas dan python.",
    "saat magang tugas saya adalah menulis query sql untuk menarik data bulanan dari mysql.",
    "saya menganalisis sentimen ulasan aplikasi menggunakan naive bayes di jupyter notebook.",

    # KELAS 2: STRONG (Struktur STAR lengkap, ada tools, kontribusi jelas, & hasil terukur)
    "saya mengoptimasi query postgresql sehingga waktu penarikan data laporan berkurang dari 2 jam menjadi 15 menit.",
    "saya membangun dashboard interactif di power bi yang membantu tim marketing menaikkan konversi penjualan sebesar 12 persen.",
    "menggunakan python dan pandas saya membersihkan 10 ribu baris data sehingga akurasi prediksi model naik menjadi 95 persen.",
    "saya menemukan anomali transaksi menggunakan exploratory data analysis atau EDA sehingga perusahaan terhindar dari rugi finansial."
]

# Label: 0 = Weak, 1 = Average, 2 = Strong
label_train = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])

# Konversi teks menjadi Tensor String murni agar tidak terkena error str3968
X_train = tf.constant(jawaban_train, dtype=tf.string)
y_train = label_train

# =====================================================================
# 2. PROSES TEXT VECTORIZATION (ADAPT DATA)
# =====================================================================
vectorize_layer = TextVectorization(max_tokens=1000, output_mode='int', output_sequence_length=20)
vectorize_layer.adapt(X_train)

# =====================================================================
# 3. MEMBANGUN MODEL MENGGUNAKAN TENSORFLOW FUNCTIONAL API (MAIN QUEST)
# =====================================================================
# Menentukan pintu masuk input teks secara eksplisit
inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="input_teks_jawaban")

# Menghubungkan layer secara fungsional
x = vectorize_layer(inputs)
x = Embedding(input_dim=1000, output_dim=16, name="embedding_layer")(x)
x = GlobalAveragePooling1D(name="pooling_layer")(x)
x = Dense(16, activation='relu', name="dense_hidden")(x)
outputs = Dense(3, activation='softmax', name="dense_output")(inputs=x)

# Inisialisasi model Functional
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Road2Work_Scoring_Model")

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# =====================================================================
# 4. IMPLEMENTASI COMPONENT KUSTOM: CUSTOM CALLBACK (MAIN QUEST)
# =====================================================================
class TargetAccuracyCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Menghentikan training jika akurasi sudah mencapai target minimum checklist (85%)
        if logs.get('accuracy') >= 0.90:
            print(f"\n🎉 [Custom Callback] Akurasi telah mencapai {logs.get('accuracy')*100:.2f}%. Target Kelulusan Checklist Terpenuhi!")
            self.model.stop_training = True

custom_callback = TargetAccuracyCallback()

# =====================================================================
# 5. TRAINING MODEL (MEMENUHI METRIK AKURASI SIDE QUEST)
# =====================================================================
print("\n--- Memulai Proses Pelatihan Model ---")
history = model.fit(X_train, y_train, epochs=30, callbacks=[custom_callback], verbose=1)

# =====================================================================
# 6. EXPORT MODEL KE FORMAT PRODUCTION .KERAS (MAIN QUEST)
# =====================================================================
print("\n--- Menyimpan dan Mengekspor Model ---")
model.save("road2work_scoring_model.keras")
print("💾 File 'road2work_scoring_model.keras' berhasil di-generate di sistem!")

# =====================================================================
# 7. UJI PREDIKSI JAWABAN BARU
# =====================================================================
print("\n--- Pengujian Inferensi Real-time ---")
teks_test = tf.constant(["saya mengoptimasi database menggunakan sql dan membuat script otomatisasi dengan python sehingga mempercepat efisiensi laporan harian tim sebesar 20 persen"], dtype=tf.string)
prediksi = model.predict(teks_test)
kelas_prediksi = np.argmax(prediksi)

label_status = ["Weak (Jelek)", "Average (Sedang)", "Strong (Bagus)"]
print(f"Teks Input: '{teks_test.numpy()[0].decode('utf-8')}'")
print(f"Hasil Analisis Model TensorFlow: Kategori -> {label_status[kelas_prediksi]}")

TensorFlow Version: 2.20.0

--- Memulai Proses Pelatihan Model ---
Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4167 - loss: 1.0987
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.6667 - loss: 1.0975
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.7500 - loss: 1.0966
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8333 - loss: 1.0956
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9167 - loss: 1.0948
🎉 [Custom Callback] Akurasi telah mencapai 91.67%. Target Kelulusan Checklist Terpenuhi!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9167 - loss: 1.0948

--- Menyimpan dan Mengekspor Model ---
💾 File 'road2work_scoring_model.keras' berhasil di-generate di sistem!

--- Pengujian Inferensi Real-time ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Teks Input: 'saya mengoptimasi database menggunakan sql dan membuat script otomatisasi dengan python sehingga mempercepat efisiensi laporan harian tim sebesar 20 persen'
